# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, overview, and analyze the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and the Croissant metadata schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- Title: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
- URL: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print('Dataset Title:', metadata.name)
print('\nDescription:', metadata.description)
print('\nFields:', ', '.join(metadata.__dict__.keys()))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and fields, referencing their `@id`s
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print('No record sets found in the top-level `recordSet` property. Searching for all record sets in files...')
    # Sometimes, Croissant schemas list records in the 'distribution' files
    record_sets_found = []
    distributions = getattr(metadata, 'distribution', [])
    from collections.abc import Iterable
    for dist in distributions:
        # Try to access contained record sets inside distribution/files
        recs = getattr(dist, 'recordSet', [])
        if isinstance(recs, Iterable):
            for rec in recs:
                rec_id = getattr(rec, '@id', None)
                print(f"- RecordSet @id: {rec_id if rec_id else rec}")
                record_sets_found.append(rec_id if rec_id else rec)
        elif recs:
            rec_id = getattr(recs, '@id', None)
            print(f"- RecordSet @id: {rec_id if rec_id else recs}")
            record_sets_found.append(rec_id if rec_id else recs)
    # collect record set IDs
    record_sets = list(filter(None, record_sets_found))
else:
    for rec in record_sets:
        rec_id = getattr(rec, '@id', None)
        print(f"- RecordSet @id: {rec_id if rec_id else rec}")
    # collect record set IDs
    record_sets = [getattr(rec, '@id', rec) for rec in record_sets]

# Get fields for each record set
if record_sets:
    for rs_id in record_sets:
        print(f"\nFields in RecordSet '{rs_id}':")
        try:
            # Find record set definition
            rs_obj = None
            if hasattr(metadata, 'recordSet'):
                for r in getattr(metadata, 'recordSet'):
                    if getattr(r, '@id', None) == rs_id:
                        rs_obj = r
                        break
            if not rs_obj:
                # Try distributions
                for d in getattr(metadata, 'distribution', []):
                    for r in getattr(d, 'recordSet', []):
                        if getattr(r, '@id', None) == rs_id:
                            rs_obj = r
                            break
            if rs_obj:
                for field in getattr(rs_obj, 'field', []):
                    f_id = getattr(field, '@id', field)
                    print(f"    - Field @id: {f_id}")
            else:
                print("   (fields unavailable or not listed)")
        except Exception as e:
            print(f"   (error reading fields: {e})")
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Manually inspect which record sets are present (from previous output, example @id values):
rs_ids = record_sets # Should be a list of record set `@id`s
dataframes = {}

for rs_id in rs_ids:
    print(f"\nLoading records from RecordSet: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
            print(df.head())
        else:
            print("No records found for this record set.")
    except Exception as e:
        print(f"   Error loading records: {e}")

# Select one record set to focus on (by default, use the first if available):
if rs_ids:
    main_rs_id = rs_ids[0]
    print(f"\nMain record set for analysis: {main_rs_id}")
    if main_rs_id in dataframes:
        print(dataframes[main_rs_id].head())
    else:
        print("DataFrame for main record set is empty.")
else:
    main_rs_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Determine which numeric fields are present
df = dataframes.get(main_rs_id, pd.DataFrame())
if len(df) == 0:
    print('No records found for analysis.')
else:
    print('Available columns:', df.columns.tolist())
    # Try to pick a numeric field by heuristics
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Try to coerce object columns to numeric if possible
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                pass
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print('Numeric fields identified:', numeric_cols)

    if len(numeric_cols) > 0:
        numeric_field = numeric_cols[0]
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}):")
        print(filtered_df.head())

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Group by a likely categorical field if present
        possible_group_fields = [col for col in df.columns if df[col].dtype == 'object' and not col.startswith('Unnamed')]
        group_field = possible_group_fields[0] if possible_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
            print(grouped_df.head())
        else:
            print('No suitable group field found.')
    else:
        print('No numeric fields available for analysis.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and main_rs_id in dataframes and not dataframes[main_rs_id].empty:
    df = dataframes[main_rs_id]

    # Numeric field and group field used as above
    if 'numeric_cols' not in locals() or not numeric_cols:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]
        plt.figure(figsize=(8,5))
        sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
        plt.title(f'Distribution of {numeric_field}')
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()

        # Boxplot by group_field
        possible_group_fields = [col for col in df.columns if df[col].dtype == 'object' and not col.startswith('Unnamed')]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            plt.figure(figsize=(10,5))
            order = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).index.tolist()
            sns.boxplot(x=group_field, y=numeric_field, data=df, order=order)
            plt.title(f'{numeric_field} by {group_field}')
            plt.xticks(rotation=45)
            plt.show()
else:
    print('No main record set DataFrame available for visualization.')

## 6. Conclusion
In this notebook, we loaded the FAIR² mass spectrometry dataset using its Croissant schema and explored its record sets, fields, and records with the `mlcroissant` library. We also demonstrated simple exploratory analysis via filtering, normalization, grouping, and plotting. For an in-depth analysis, refer to domain-specific field descriptions and expand the EDA accordingly.

You can adapt this notebook for other Croissant datasets by updating the schema URL and referencing the relevant `@id`s for record sets and fields.
